In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Input and output folders
EXECUTION_DIR = Path("results/execution")
OUTPUT_DIR = Path("results/evaluation_joint")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

execution_files = sorted(EXECUTION_DIR.glob("*_execution.csv"))

print(f"Found {len(execution_files)} execution files:")
for f in execution_files:
    print("-", f.name)
    
# Use all execution CSV files by default. Replace this list to evaluate selected runs only.
selected_files = execution_files

print(f"Selected {len(selected_files)} files:")
for f in selected_files:
    print("-", f.name)



frames = []

for path in selected_files:
    df = pd.read_csv(path)
    df["source_file"] = path.name
    frames.append(df)

df_joint = pd.concat(frames, ignore_index=True)

In [ ]:
# Keep only the first occurrence of each column name
df_joint = df_joint.loc[:, ~df_joint.columns.duplicated()].copy()

print("Rows:", len(df_joint))
print("Models:", sorted(df_joint["model_name"].dropna().unique().tolist()))

run_question_counts = (
    df_joint
    .groupby(["model_name", "run_id"])["question_id"]
    .nunique()
    .reset_index(name="unique_questions")
)

run_question_counts

In [ ]:
run_summary = (
    df_joint.groupby(["model_name", "run_id"], as_index=False)
    .agg(
        n_questions=("question_id", "count"),
        exact_match_mean=("exact_match_accuracy", "mean"),
        f1_mean=("f1_score", "mean"),
        execution_success_rate=("execution_success", "mean"),
        latency_mean=("total_pipeline_latency_seconds", "mean"),
        tokens_mean=("total_tokens", "mean"),
    )
)

comparison_table = (
    run_summary.groupby("model_name", as_index=False)
    .agg(
        runs=("run_id", "nunique"),
        exact_match_mean=("exact_match_mean", "mean"),
        exact_match_std=("exact_match_mean", "std"),
        f1_mean=("f1_mean", "mean"),
        f1_std=("f1_mean", "std"),
        execution_success_mean=("execution_success_rate", "mean"),
        execution_success_std=("execution_success_rate", "std"),
        latency_mean=("latency_mean", "mean"),
        latency_std=("latency_mean", "std"),
        tokens_mean=("tokens_mean", "mean"),
        tokens_std=("tokens_mean", "std"),
    )
)

pareto_metrics = [
    ("exact_match_mean", True),          # maximize
    ("f1_mean", True),                   # maximize
    ("execution_success_mean", True),    # maximize
    ("latency_mean", False),             # minimize
    ("tokens_mean", False),              # minimize
]

def pareto_optimal_mask(df, metrics):
    work = df[[col for col, _ in metrics]].copy()

    for col, maximize in metrics:
        if maximize:
            work[col] = -work[col]

    values = work.to_numpy(dtype=float)
    n = len(values)
    is_pareto = np.ones(n, dtype=bool)

    for i in range(n):
        for j in range(n):
            if i == j:
                continue

            no_worse = np.all(values[j] <= values[i])
            strictly_better = np.any(values[j] < values[i])

            if no_worse and strictly_better:
                is_pareto[i] = False
                break

    return is_pareto

comparison_table["pareto_optimal"] = pareto_optimal_mask(comparison_table, pareto_metrics)

comparison_table_rounded = comparison_table.copy().round(3)
comparison_table_rounded["pareto_optimal"] = comparison_table_rounded["pareto_optimal"].map(
    {True: "Yes", False: "No"}
)

comparison_table_rounded

In [ ]:
# Okabe-Ito colorblind palette
colorblind_palette = [
    "#E69F00",  # orange
    "#56B4E9",  # sky blue
    "#009E73",  # bluish green
    "#CC79A7",  # pinkish purple
]

model_colors = {
    "Qwen/Qwen2.5-3B-Instruct": colorblind_palette[0],
    "Qwen/Qwen2.5-7B-Instruct": colorblind_palette[1],
    "Qwen/Qwen2.5-Coder-3B-Instruct": colorblind_palette[2],
    "Qwen/Qwen2.5-Coder-7B-Instruct": colorblind_palette[3],
}

In [ ]:
outcome_counts = (
    df_joint.groupby(["model_name", "outcome_type"])
    .size()
    .unstack(fill_value=0)
)

outcome_counts.plot(
    kind="bar",
    stacked=False,
    figsize=(9, 5),
    color=colorblind_palette[:len(outcome_counts.columns)]
)

plt.title("Outcome counts per model")
plt.xlabel("")
plt.ylabel("Count (n = 520 per model)")
plt.ylim(0, 450)
plt.xticks(rotation=10, ha="right")
plt.legend(title="Outcome type", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "outcome_type_grouped_bar.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Combined effectiveness chart used in presentation
effectiveness_metrics = [
    ("exact_match_mean", "Exact match"),
    ("f1_mean", "F1-score"),
    ("execution_success_mean", "Execution success"),
]

x = np.arange(len(comparison_table["model_name"]))
width = 0.25

plt.figure(figsize=(9, 5))

for i, (col, label) in enumerate(effectiveness_metrics):
    plt.bar(
        x + (i - 1) * width,
        comparison_table[col],
        width=width,
        label=label,
        color=colorblind_palette[i],
    )

plt.xticks(x, comparison_table["model_name"], rotation=10)
plt.ylabel("Mean metric value (0-1)")
plt.title("Effectiveness metrics by model")
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "effectiveness_bar.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Separate effectiveness charts used in the thesis report
effectiveness_metrics = [
    ("exact_match_mean", "Exact match", "exact_match_bar.png"),
    ("f1_mean", "F1-score", "f1_bar.png"),
    ("execution_success_mean", "Execution success rate", "execution_success_bar.png"),
]

for col, title, filename in effectiveness_metrics:
    plt.figure(figsize=(7, 5))
    plt.bar(comparison_table["model_name"], comparison_table[col])
    plt.xticks(rotation=10)
    plt.ylabel(f"{title} (0-1)")
    plt.title(f"Mean {title.lower()} by model")
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / filename, dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
# Efficiency charts used in the thesis report
plt.figure(figsize=(7, 5))
plt.bar(comparison_table["model_name"], comparison_table["latency_mean"])
plt.xticks(rotation=10)
plt.ylabel("Latency (seconds)")
plt.title("Mean latency by model")
plt.ylim(0,10)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "latency_bar.png", dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(7, 5))
plt.bar(comparison_table["model_name"], comparison_table["tokens_mean"])
plt.xticks(rotation=10)
plt.ylabel("Token usage (tokens)")
plt.title("Mean token usage by model")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "tokens_bar.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
def boxplot_by_model(run_summary, metric_col, title, ylabel, filename, ylim=None):
    model_groups = list(run_summary.groupby("model_name"))

    grouped = [
        group[metric_col].dropna().values
        for _, group in model_groups
    ]

    labels = [
        name
        for name, _ in model_groups
    ]

    plt.figure(figsize=(7, 5))
    plt.boxplot(grouped, tick_labels=labels)
    plt.xticks(rotation=10)
    plt.title(title)
    plt.ylabel(ylabel)

    if ylim is not None:
        plt.ylim(ylim)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / filename, dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
def run_scatter(
    run_summary, x_col, y_col, xlabel, ylabel, title, filename
):
    plt.figure(figsize=(7, 5))

    offsets = [(5, 5), (5, -10), (-20, 5), (-20, -10), (10, 12), (10, -15)]

    for model_name, group in run_summary.groupby("model_name"):
        plt.scatter(group[x_col], group[y_col], label=model_name, color=model_colors.get(model_name, "#999999"))

        for i, (_, row) in enumerate(group.iterrows()):
            dx, dy = offsets[i % len(offsets)]
            short_label = str(row["run_id"]).split("_")[-1]

            plt.annotate(
                short_label,
                (row[x_col], row[y_col]),
                xytext=(dx, dy),
                textcoords="offset points",
                fontsize=10,
                alpha=0.75,
                bbox=dict(
                    boxstyle="round,pad=0.15",
                    facecolor="white",
                    edgecolor="none",
                    alpha=0.6
                )
            )

    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / filename, dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
boxplot_by_model(
    run_summary,
    metric_col="f1_mean",
    title="Mean F1-score across runs",
    ylabel="F1-score",
    filename="f1_boxplot.png",
)
boxplot_by_model(
    run_summary,
    metric_col="exact_match_mean",
    title="Mean exact match across runs",
    ylabel="Exact match",
    filename="exact_match_boxplot.png",
)
boxplot_by_model(
    run_summary,
    metric_col="execution_success_rate",
    title="Mean execution success rate across runs",
    ylabel="Execution success",
    filename="execution_success_boxplot.png",
)
boxplot_by_model(
    run_summary,
    metric_col="latency_mean",
    title="Mean latency across runs",
    ylabel="Latency (seconds)",
    filename="latency_boxplot.png",
    ylim=(0,10)
)
boxplot_by_model(
    run_summary,
    metric_col="tokens_mean",
    title="Mean token usage across runs",
    ylabel="Token usage (tokens)",
    filename="tokens_boxplot.png",
)
run_scatter(
    run_summary,
    "latency_mean",
    "f1_mean",
    "Mean latency per run (seconds)",
    "Mean F1-score per run",
    "Latency vs. F1-score per run",
    "run_latency_vs_f1_scatter.png"
)

run_scatter(
    run_summary,
    "tokens_mean",
    "f1_mean",
    "Mean token usage per run (tokens)",
    "Mean F1-score per run",
    "Token usage vs. F1-score per run",
    "run_tokens_vs_f1_scatter.png"
)

run_scatter(
    run_summary,
    "latency_mean",
    "execution_success_rate",
    "Mean latency per run (seconds)",
    "Execution success rate per run",
    "Latency vs. execution success per run",
    "run_latency_vs_execution_scatter.png"
)

run_scatter(
    run_summary,
    "tokens_mean",
    "execution_success_rate",
    "Mean token usage per run (tokens)",
    "Execution success rate per run",
    "Token usage vs. execution success per run",
    "run_tokens_vs_execution_scatter.png"
)

In [ ]:
run_summary.to_csv(OUTPUT_DIR / "run_summary.csv", index=False)
comparison_table_rounded.to_csv(OUTPUT_DIR / "comparison_table.csv", index=False)

print("Saved to:", OUTPUT_DIR)